In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("/content/drive/MyDrive/Text summarizer/samsum-train.csv")
val_data=pd.read_csv("/content/drive/MyDrive/Text summarizer/samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
# random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
val_data.shape

(500, 3)

# **Data** **Pre-processing**

In [ ]:
import re

In [ ]:
def clean_text(text):
  text=re.sub(r"\r\n"," ",text) #remove nextline
  text=re.sub(r"\s+"," ",text) #remove space
  text=re.sub(r"<.*?>"," ",text) #remove html tags
  text=re.sub(r"[^a-zA-Z]"," ",text) #remove non-alphabetic characters
  text=text.strip().lower()
  return text

In [ ]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_text)
train_data["summary"]=train_data["summary"].apply(clean_text)

val_data["dialogue"]=val_data["dialogue"].apply(clean_text)
val_data["summary"]=val_data["summary"].apply(clean_text)

# **Tokenize**

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# rawdata => tokens

def tokenize(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 7102, 3, 23, 764, 640, 48, 403, 17, 77, 3, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 3, 7997, 15, 7102, 2049, 68, 3, 23, 3, 162, 641, 608, 34, 3, 7997, 15, 68, 2049, 21, 1631, 81, 140, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
len(train_dataset[0]["input_ids"])

512

# **Working with our model**

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# set device for training
import torch

if torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")

print("device:",device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
import os

project_dir = "/content/drive/MyDrive/Text summarizer"

os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print(os.getcwd())

/content/drive/MyDrive/Text summarizer


In [ ]:
# Training Argument
training_arg=TrainingArguments(
    output_dir=os.path.join(project_dir, "results"),

    num_train_epochs=10,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500

)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_arg,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train the mode
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.899154,0.403739
2,0.421498,0.377714
3,0.393819,0.369291
4,0.379296,0.363943
5,0.368208,0.361875
6,0.361226,0.359590
7,0.355435,0.357793
8,0.351218,0.356990
9,0.348465,0.356792
10,0.346614,0.356762


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5000, training_loss=0.422493310546875, metrics={'train_runtime': 2105.3223, 'train_samples_per_second': 18.999, 'train_steps_per_second': 2.375, 'total_flos': 5413672058880000.0, 'train_loss': 0.422493310546875, 'epoch': 10.0})

In [ ]:
import os

save_path = "/content/drive/MyDrive/Text summarizer/save_summary_model"

os.makedirs(save_path, exist_ok=True)

# Save model
model.save_pretrained(save_path)

# Save tokenizer
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: /content/drive/MyDrive/Text summarizer/save_summary_model


In [ ]:
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Text summarizer/save_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/Text summarizer/save_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# **Core logic for summarization**

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  Expert: Companies are investing heavily in machine learning systems to automate tasks and improve decision-making. experts highlight the importance of responsible AI development
